# MoCapGate × GVHMR — мокап из видео на GPU Colab

**Видео → движение всего тела в мировых координатах (GVHMR, SIGGRAPH Asia 2024) → BVH для Blender / Maya
и файл дубля для MoCapGate Studio.** *Video → GVHMR world-grounded motion → BVH + a MoCapGate take.*

Как пользоваться:
1. **Среда выполнения → Сменить среду → GPU (T4 подойдёт)**.
2. Запустите ячейки по порядку (▶ слева или *Среда выполнения → Выполнить всё*). Первая установка — 6–10 минут,
   дальше в той же сессии — секунды.
3. На шаге 3 загрузите **SMPLX_NEUTRAL.npz** — бесплатная регистрация на
   [smpl-x.is.tue.mpg.de](https://smpl-x.is.tue.mpg.de) → *Download* → *SMPL-X v1.1 (NPZ)*. Один раз положите файл
   в Google Drive в папку `MoCapGate/` — дальше ноутбук найдёт его сам.
4. На шаге 4 загрузите видео (или возьмите пример). Результат — архив с `*.bvh` и `*.mocapgate.json`:
   BVH сразу в Blender (*Import → Motion Capture*, Scale 0.01), JSON — в MoCapGate Studio для проверки и экспорта.

Советы по съёмке: человек целиком в кадре, камера на уровне пояса, хороший свет, без сильного размытия.
Если камера стоит на месте — включите `static_camera`: так точнее.

> Лицензии: GVHMR и модель SMPL-X — **только для некоммерческого использования**. Видео обрабатывается в вашей
> сессии Colab и никуда больше не уходит.

In [ ]:
#@title Шаг 0. Настройки
max_people = 4  #@param {type:"integer"}
#@markdown ↑ максимум участников (1–8), отдельный BVH для каждого
static_camera = False  #@param {type:"boolean"}
#@markdown ↑ камера неподвижна (штатив) — точнее и быстрее
focal_mm = 0  #@param {type:"integer"}
#@markdown ↑ фокусное расстояние в мм (35-мм экв.): iPhone 0.5×/1×/2×/3× ≈ 13/24/48/77. 0 — оценить автоматически
target_fps = 30  #@param {type:"integer"}
#@markdown ↑ GVHMR обучен на 30 fps — видео приводится к этой частоте
save_to_drive = True  #@param {type:"boolean"}
#@markdown ↑ сохранить результат в Google Drive → MoCapGate/out

In [ ]:
#@title Шаг 1. Проверка GPU
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit("Нет GPU. Среда выполнения → Сменить среду выполнения → T4 GPU, затем запустите ячейки заново.")
print("GPU:", out.stdout.strip())

In [ ]:
#@title Шаг 2. Установка GVHMR (один раз за сессию, 6–10 мин)
import os, subprocess, pathlib
GV = pathlib.Path("/content/GVHMR"); VENV = pathlib.Path("/content/gvhmr-venv"); PY = VENV / "bin/python"
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)
if not (GV / ".ok").exists():
    sh("pip -q install uv")
    if not GV.exists():
        sh(f"git clone -q https://github.com/zju3dv/GVHMR {GV} && git -C {GV} checkout -q ee960bb6e2ea2d381aa97f08e9b71ef320b624b1")
    sh(f"uv venv -q -p 3.10 {VENV}")
    uv = f"uv pip install -q -p {PY}"
    sh(f"{uv} torch==2.3.0+cu121 torchvision==0.18.0+cu121 --index-url https://download.pytorch.org/whl/cu121")
    sh(f"{uv} numpy==1.23.5 cython setuptools wheel")
    sh(f"{uv} timm==0.9.12 lightning==2.3.0 hydra-core==1.3 hydra-zen hydra_colorlog rich matplotlib "
       f"tensorboardX opencv-python ffmpeg-python scikit-image termcolor einops imageio==2.34.1 av==13.0.0 joblib "
       f"trimesh smplx wis3d pycolmap ultralytics==8.2.42 lapx yacs")
    sh(f"{uv} --no-build-isolation cython_bbox")
    sh(f"{uv} https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/"
       f"pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl")
    sh(f"{uv} --no-deps -e {GV}")
    ck = GV / "inputs/checkpoints"
    for rel in ("gvhmr/gvhmr_siga24_release.ckpt", "hmr2/epoch=10-step=25000.ckpt",
                "vitpose/vitpose-h-multi-coco.pth", "yolo/yolov8x.pt"):
        dst = ck / rel
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            sh(f"curl -sSfL -o '{dst}' 'https://huggingface.co/camenduru/GVHMR/resolve/main/{rel.replace('=', '%3D')}'")
    (GV / ".ok").touch()
print("GVHMR готов:", GV)

In [ ]:
#@title Ядро MoCapGate (SMPL-X → BVH)
import os; os.makedirs('/content/mocapgate_core', exist_ok=True)

In [ ]:
%%writefile /content/mocapgate_core/__init__.py

In [ ]:
%%writefile /content/mocapgate_core/rotations.py
"""MoCapGate — математика поворотов на stdlib (без numpy).

Матрицы 3×3 — кортежи строк. Эйлер в порядке BVH-каналов ZXY:
R = Rz(z) · Rx(x) · Ry(y), углы в градусах.
"""
from __future__ import annotations
import math

Mat = tuple[tuple[float, float, float], ...]


def axis_angle_to_matrix(v) -> Mat:
    """Rodrigues: вектор оси, умноженный на угол (рад) → матрица поворота."""
    x, y, z = float(v[0]), float(v[1]), float(v[2])
    a = math.sqrt(x * x + y * y + z * z)
    if a < 1e-12:
        return ((1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.0, 1.0))
    x, y, z = x / a, y / a, z / a
    c, s = math.cos(a), math.sin(a)
    t = 1 - c
    return (
        (t * x * x + c, t * x * y - s * z, t * x * z + s * y),
        (t * x * y + s * z, t * y * y + c, t * y * z - s * x),
        (t * x * z - s * y, t * y * z + s * x, t * z * z + c),
    )


def matmul(a: Mat, b: Mat) -> Mat:
    return tuple(
        tuple(sum(a[i][k] * b[k][j] for k in range(3)) for j in range(3)) for i in range(3)
    )  # type: ignore[return-value]


def mat_vec(m: Mat, v) -> tuple[float, float, float]:
    return tuple(sum(m[i][k] * v[k] for k in range(3)) for i in range(3))  # type: ignore[return-value]


def euler_zxy_to_matrix(z: float, x: float, y: float) -> Mat:
    z, x, y = math.radians(z), math.radians(x), math.radians(y)
    cz, sz, cx, sx, cy, sy = math.cos(z), math.sin(z), math.cos(x), math.sin(x), math.cos(y), math.sin(y)
    rz = ((cz, -sz, 0.0), (sz, cz, 0.0), (0.0, 0.0, 1.0))
    rx = ((1.0, 0.0, 0.0), (0.0, cx, -sx), (0.0, sx, cx))
    ry = ((cy, 0.0, sy), (0.0, 1.0, 0.0), (-sy, 0.0, cy))
    return matmul(matmul(rz, rx), ry)


def matrix_to_euler_zxy(m: Mat) -> tuple[float, float, float]:
    """Матрица → (z, x, y) в градусах для R = Rz·Rx·Ry.

    Из разложения: m[2][1] = sin x, m[0][1] = -sin z·cos x, m[1][1] = cos z·cos x,
    m[2][0] = -cos x·sin y, m[2][2] = cos x·cos y.
    """
    sx = max(-1.0, min(1.0, m[2][1]))
    x = math.asin(sx)
    if abs(sx) < 0.999999:
        z = math.atan2(-m[0][1], m[1][1])
        y = math.atan2(-m[2][0], m[2][2])
    else:  # gimbal lock: z и y вырождаются в один угол, кладём всё в z
        y = 0.0
        z = math.atan2(m[1][0], m[0][0])
    return math.degrees(z), math.degrees(x), math.degrees(y)


def _wrap_near(a: float, ref: float) -> float:
    return a + 360.0 * round((ref - a) / 360.0)


def continuous_euler(zxy: tuple[float, float, float], prev: tuple[float, float, float] | None):
    """Выбрать эквивалентную тройку углов, ближайшую к предыдущему кадру.

    Без этого кривые в Blender/Maya дают скачки на ±360° и «кувырки» при
    интерполяции. Рассматриваем обе ветви решения ZXY (x и 180−x) и все
    сдвиги на 360°.
    """
    if prev is None:
        return zxy
    z, x, y = zxy
    best, best_d = zxy, float("inf")
    for cz, cx, cy in ((z, x, y), (z + 180.0, 180.0 - x, y + 180.0)):
        cand = (_wrap_near(cz, prev[0]), _wrap_near(cx, prev[1]), _wrap_near(cy, prev[2]))
        d = sum((c - p) ** 2 for c, p in zip(cand, prev))
        if d < best_d:
            best, best_d = cand, d
    return best


# ---------------------------------------------------------------- кватернионы (w, x, y, z)

def matrix_to_quat(m: Mat) -> tuple[float, float, float, float]:
    tr = m[0][0] + m[1][1] + m[2][2]
    if tr > 0:
        s = math.sqrt(tr + 1.0) * 2
        q = (0.25 * s, (m[2][1] - m[1][2]) / s, (m[0][2] - m[2][0]) / s, (m[1][0] - m[0][1]) / s)
    elif m[0][0] > m[1][1] and m[0][0] > m[2][2]:
        s = math.sqrt(1.0 + m[0][0] - m[1][1] - m[2][2]) * 2
        q = ((m[2][1] - m[1][2]) / s, 0.25 * s, (m[0][1] + m[1][0]) / s, (m[0][2] + m[2][0]) / s)
    elif m[1][1] > m[2][2]:
        s = math.sqrt(1.0 + m[1][1] - m[0][0] - m[2][2]) * 2
        q = ((m[0][2] - m[2][0]) / s, (m[0][1] + m[1][0]) / s, 0.25 * s, (m[1][2] + m[2][1]) / s)
    else:
        s = math.sqrt(1.0 + m[2][2] - m[0][0] - m[1][1]) * 2
        q = ((m[1][0] - m[0][1]) / s, (m[0][2] + m[2][0]) / s, (m[1][2] + m[2][1]) / s, 0.25 * s)
    n = math.sqrt(sum(c * c for c in q))
    return tuple(c / n for c in q)  # type: ignore[return-value]


def quat_to_matrix(q) -> Mat:
    w, x, y, z = q
    return (
        (1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)),
        (2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)),
        (2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)),
    )


def slerp(a, b, t: float):
    d = sum(p * q for p, q in zip(a, b))
    if d < 0:
        b, d = tuple(-c for c in b), -d
    if d > 0.9995:
        q = tuple(p + t * (r - p) for p, r in zip(a, b))
    else:
        th = math.acos(d)
        s = math.sin(th)
        wa, wb = math.sin((1 - t) * th) / s, math.sin(t * th) / s
        q = tuple(wa * p + wb * r for p, r in zip(a, b))
    n = math.sqrt(sum(c * c for c in q))
    return tuple(c / n for c in q)


def matrix_to_axis_angle(m: Mat) -> list[float]:
    w, x, y, z = matrix_to_quat(m)
    if w < 0:
        w, x, y, z = -w, -x, -y, -z
    s = math.sqrt(x * x + y * y + z * z)
    if s < 1e-12:
        return [0.0, 0.0, 0.0]
    a = 2 * math.atan2(s, w)
    return [x / s * a, y / s * a, z / s * a]


def transpose(m: Mat) -> Mat:
    return tuple(tuple(m[j][i] for j in range(3)) for i in range(3))  # type: ignore[return-value]


def frame_from_axes(x, y) -> Mat:
    """Ортонормированный базис: X по x, Y — y без проекции на X, Z = X×Y. Столбцы = оси."""
    xn = _norm(x)
    d = sum(a * b for a, b in zip(y, xn))
    yn = _norm([y[i] - d * xn[i] for i in range(3)])
    zn = _cross(xn, yn)
    return tuple(tuple((xn, yn, zn)[c][r] for c in range(3)) for r in range(3))  # type: ignore[return-value]


def swing(a, b) -> Mat:
    """Кратчайший поворот, переводящий направление a в направление b."""
    a, b = _norm(a), _norm(b)
    c = _cross(a, b)
    s = math.sqrt(sum(v * v for v in c))
    d = sum(p * q for p, q in zip(a, b))
    if s < 1e-9:
        if d > 0:
            return ((1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.0, 1.0))
        perp = _cross(a, (1.0, 0.0, 0.0) if abs(a[0]) < 0.9 else (0.0, 1.0, 0.0))
        return axis_angle_to_matrix([v * math.pi for v in _norm(perp)])
    ang = math.atan2(s, d)
    return axis_angle_to_matrix([v / s * ang for v in c])


def _norm(v):
    n = math.sqrt(sum(c * c for c in v)) or 1e-12
    return tuple(c / n for c in v)


def _cross(a, b):
    return (a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0])

In [ ]:
%%writefile /content/mocapgate_core/skeleton.py
"""MoCapGate — гуманоидный скелет для BVH (иерархия суставов + оффсеты).

Оффсеты в сантиметрах, примерные пропорции взрослого человека. Этого достаточно
для валидного BVH, который открывают Blender и Maya; точные пропорции и ретаргет
на конкретный риг — на этапе доводки.
"""
from __future__ import annotations
from dataclasses import dataclass, field


@dataclass
class Joint:
    name: str
    offset: tuple[float, float, float]
    children: list["Joint"] = field(default_factory=list)
    end_offset: tuple[float, float, float] | None = None  # End Site для листьев


def humanoid() -> Joint:
    """Стандартный гуманоид: Hips → позвоночник/голова, руки, ноги."""
    def J(n, off, ch=None, end=None):
        return Joint(n, off, ch or [], end)
    return J("Hips", (0, 0, 0), [
        J("Spine", (0, 10, 0), [
            J("Spine1", (0, 12, 0), [
                J("Neck", (0, 14, 0), [
                    J("Head", (0, 6, 0), end=(0, 10, 0)),
                ]),
                J("LeftShoulder", (6, 12, 0), [
                    J("LeftArm", (13, 0, 0), [
                        J("LeftForeArm", (26, 0, 0), [
                            J("LeftHand", (24, 0, 0), end=(10, 0, 0)),
                        ]),
                    ]),
                ]),
                J("RightShoulder", (-6, 12, 0), [
                    J("RightArm", (-13, 0, 0), [
                        J("RightForeArm", (-26, 0, 0), [
                            J("RightHand", (-24, 0, 0), end=(-10, 0, 0)),
                        ]),
                    ]),
                ]),
            ]),
        ]),
        J("LeftUpLeg", (9, -2, 0), [
            J("LeftLeg", (0, -42, 0), [
                J("LeftFoot", (0, -42, 0), end=(0, -4, 14)),
            ]),
        ]),
        J("RightUpLeg", (-9, -2, 0), [
            J("RightLeg", (0, -42, 0), [
                J("RightFoot", (0, -42, 0), end=(0, -4, 14)),
            ]),
        ]),
    ])


def joint_names(root: Joint) -> list[str]:
    names: list[str] = []
    def walk(j: Joint):
        names.append(j.name)
        for c in j.children:
            walk(c)
    walk(root)
    return names

In [ ]:
%%writefile /content/mocapgate_core/bvh.py
"""MoCapGate — запись BVH (универсальный формат мокапа для Blender и Maya).

Строит секцию HIERARCHY из скелета и MOTION из списка кадров. Корень несёт
позицию+поворот (6 каналов), остальные суставы — поворот (3 канала, порядок ZXY).
Кадр: {"Hips": {"pos": (x,y,z), "rot": (z,x,y)}, "<joint>": {"rot": (z,x,y)}}.
Отсутствующие повороты считаются нулевыми.
"""
from __future__ import annotations
from mocapgate_core.skeleton import Joint

_ROT = ("Zrotation", "Xrotation", "Yrotation")
_POS = ("Xposition", "Yposition", "Zposition")


def _hierarchy(root: Joint) -> tuple[list[str], list[tuple[str, str]]]:
    lines: list[str] = ["HIERARCHY"]
    channels: list[tuple[str, str]] = []

    def fmt(off):
        return f"{off[0]:g} {off[1]:g} {off[2]:g}"

    def walk(j: Joint, depth: int, is_root: bool):
        pad = "  " * depth
        tag = "ROOT" if is_root else "JOINT"
        lines.append(f"{pad}{tag} {j.name}")
        lines.append(f"{pad}{{")
        lines.append(f"{pad}  OFFSET {fmt(j.offset)}")
        if is_root:
            lines.append(f"{pad}  CHANNELS 6 {' '.join(_POS + _ROT)}")
            channels.extend((j.name, c) for c in _POS + _ROT)
        else:
            lines.append(f"{pad}  CHANNELS 3 {' '.join(_ROT)}")
            channels.extend((j.name, c) for c in _ROT)
        for c in j.children:
            walk(c, depth + 1, False)
        if j.end_offset is not None:
            lines.append(f"{pad}  End Site")
            lines.append(f"{pad}  {{")
            lines.append(f"{pad}    OFFSET {fmt(j.end_offset)}")
            lines.append(f"{pad}  }}")
        lines.append(f"{pad}}}")

    walk(root, 0, True)
    return lines, channels


def write(root: Joint, frames: list[dict], frame_time: float = 1 / 30) -> str:
    lines, channels = _hierarchy(root)
    out = list(lines)
    out.append("MOTION")
    out.append(f"Frames: {len(frames)}")
    out.append(f"Frame Time: {frame_time:.7f}")
    for fr in frames:
        row: list[str] = []
        for joint, chan in channels:
            data = fr.get(joint, {})
            if chan in _POS:
                i = _POS.index(chan)
                row.append(f"{data.get('pos', (0, 0, 0))[i]:.4f}")
            else:
                i = _ROT.index(chan)
                row.append(f"{data.get('rot', (0, 0, 0))[i]:.4f}")
        out.append(" ".join(row))
    return "\n".join(out) + "\n"


def save(path, text: str) -> None:
    """Записать BVH с переводами строк LF на любой ОС (на Windows write_text дал бы CRLF)."""
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(text)

In [ ]:
%%writefile /content/mocapgate_core/smpl.py
"""MoCapGate — скелет тела SMPL / SMPL-X (22 сустава корпуса) для BVH.

GVHMR, WHAM, TRAM и GEM выдают движение в параметрах SMPL(-X): локальные
повороты суставов (axis-angle) относительно T-позы, у которой оси всех суставов
совпадают с мировыми. Поэтому BVH-скелет строится прямо из rest-позиций суставов
SMPL, и поворот SMPL переходит в BVH-канал без ретаргета.

Имена суставов — в стиле Mixamo: их понимают авто-ретаргет Blender (Rokoko,
Auto-Rig Pro) и HumanIK в Maya.

Rest-позиции:
- точные — из модели SMPLX_NEUTRAL.npz с учётом betas человека (нужен numpy и
  файл модели с https://smpl-x.is.tue.mpg.de, лицензия некоммерческая);
- без модели — встроенные приблизительные значения нейтральной SMPL-X
  (повороты всё равно корректны, слегка отличаются только длины костей).
"""
from __future__ import annotations
from mocapgate_core.skeleton import Joint

# Порядок суставов SMPL-X body (первые 22 совпадают с SMPL).
NAMES = (
    "Hips", "LeftUpLeg", "RightUpLeg", "Spine", "LeftLeg", "RightLeg", "Spine1",
    "LeftFoot", "RightFoot", "Spine2", "LeftToeBase", "RightToeBase", "Neck",
    "LeftShoulder", "RightShoulder", "Head", "LeftArm", "RightArm", "LeftForeArm",
    "RightForeArm", "LeftHand", "RightHand",
)
PARENTS = (-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19)
N_JOINTS = len(NAMES)

# Приблизительные rest-позиции нейтральной SMPL-X, метры, Y вверх, лицом к +Z.
DEFAULT_REST = (
    (0.003, -0.351, 0.012),    # Hips (pelvis)
    (0.061, -0.444, -0.014),   # LeftUpLeg
    (-0.060, -0.455, -0.009),  # RightUpLeg
    (0.000, -0.240, -0.015),   # Spine
    (0.116, -0.823, -0.006),   # LeftLeg (knee)
    (-0.104, -0.818, -0.003),  # RightLeg
    (0.010, -0.104, 0.006),    # Spine1
    (0.073, -1.237, -0.044),   # LeftFoot (ankle)
    (-0.089, -1.229, -0.043),  # RightFoot
    (-0.001, -0.055, 0.026),   # Spine2
    (0.120, -1.290, 0.083),    # LeftToeBase
    (-0.125, -1.285, 0.088),   # RightToeBase
    (-0.014, 0.110, -0.018),   # Neck
    (0.045, 0.029, 0.000),     # LeftShoulder (collar)
    (-0.049, 0.028, -0.006),   # RightShoulder
    (0.011, 0.268, -0.005),    # Head
    (0.171, 0.062, -0.021),    # LeftArm (shoulder)
    (-0.179, 0.064, -0.023),   # RightArm
    (0.420, 0.038, -0.046),    # LeftForeArm (elbow)
    (-0.423, 0.036, -0.047),   # RightForeArm
    (0.675, 0.038, -0.048),    # LeftHand (wrist)
    (-0.678, 0.040, -0.047),   # RightHand
)

# End Site для листьев, метры (в системе rest-позы).
_END_SITES = {
    "Head": (0.0, 0.16, 0.0),
    "LeftToeBase": (0.0, 0.0, 0.06),
    "RightToeBase": (0.0, 0.0, 0.06),
    "LeftHand": (0.09, 0.0, 0.0),
    "RightHand": (-0.09, 0.0, 0.0),
}


def rest_from_model(model_path: str, betas=None) -> list[tuple[float, float, float]]:
    """Точные rest-позиции 22 суставов из SMPLX_NEUTRAL.npz (или SMPL .npz) с учётом betas."""
    import numpy as np  # type: ignore

    m = np.load(model_path, allow_pickle=True)
    v = np.asarray(m["v_template"], dtype=np.float64)
    if betas is not None:
        b = np.asarray(betas, dtype=np.float64).reshape(-1)
        dirs = np.asarray(m["shapedirs"], dtype=np.float64)[:, :, : len(b)]
        v = v + dirs @ b
    jreg = m["J_regressor"]
    if hasattr(jreg, "toarray"):  # scipy.sparse в некоторых сборках модели
        jreg = jreg.toarray()
    joints = np.asarray(jreg, dtype=np.float64) @ v
    return [tuple(map(float, j)) for j in joints[:N_JOINTS]]


def build_skeleton(rest, scale: float = 100.0) -> Joint:
    """Дерево Joint из rest-позиций (метры) → оффсеты в BVH-единицах (по умолчанию см)."""
    joints = [Joint(n, (0.0, 0.0, 0.0)) for n in NAMES]
    for i, p in enumerate(PARENTS):
        if p < 0:
            continue
        joints[i].offset = tuple((rest[i][k] - rest[p][k]) * scale for k in range(3))  # type: ignore[assignment]
        joints[p].children.append(joints[i])
    for name, end in _END_SITES.items():
        joints[NAMES.index(name)].end_offset = tuple(e * scale for e in end)  # type: ignore[assignment]
    return joints[0]


def fk(global_orient, body_pose, transl, rest=DEFAULT_REST):
    """Мировые позиции 22 суставов (метры) одного кадра по формулам SMPL: таз = rest[0] + transl."""
    from mocapgate_core.rotations import axis_angle_to_matrix, mat_vec, matmul

    rv = [global_orient] + [body_pose[3 * i:3 * i + 3] for i in range(N_JOINTS - 1)]
    G: list = [None] * N_JOINTS
    P: list = [None] * N_JOINTS
    for i, p in enumerate(PARENTS):
        R = axis_angle_to_matrix(rv[i])
        if p < 0:
            G[i] = R
            P[i] = tuple(rest[0][k] + float(transl[k]) for k in range(3))
        else:
            G[i] = matmul(G[p], R)
            off = mat_vec(G[p], [rest[i][k] - rest[p][k] for k in range(3)])
            P[i] = tuple(P[p][k] + off[k] for k in range(3))
    return P

In [ ]:
%%writefile /content/mocapgate_core/smpl_bvh.py
"""MoCapGate — движение SMPL(-X) → кадры BVH.

Вход — параметры одной последовательности (как у GVHMR/WHAM/TRAM/GEM):
    global_orient  L×3   axis-angle корня (рад)
    body_pose      L×63  axis-angle 21 сустава корпуса (или L×69 у SMPL — берём первые 63)
    transl         L×3   смещение корня (метры)
    betas          10    форма тела (опц., нужна только для точных длин костей)

В SMPL позиция таза в мире = rest(таз) + transl, а поворот каждого сустава
задан в системе родителя при единичных осях rest-позы. Ровно так же устроен
BVH, поэтому каждый поворот переводится в Эйлер ZXY напрямую.

params["cam_to_yup"] = True — вход в камерной системе OpenCV (Y вниз, Z от
камеры, режим incam): весь мир поворачиваем на 180° вокруг X.
"""
from __future__ import annotations
from mocapgate_core import smpl
from mocapgate_core.rotations import axis_angle_to_matrix, continuous_euler, mat_vec, matmul, matrix_to_euler_zxy
from mocapgate_core.skeleton import Joint


def mean_betas(betas) -> list[float] | None:
    """betas бывают на кадр (L×10) — усредняем; форма тела в ролике одна."""
    if betas is None or len(betas) == 0:
        return None
    if isinstance(betas[0], (int, float)):
        return [float(b) for b in betas]
    n = len(betas)
    return [sum(float(row[k]) for row in betas) / n for k in range(len(betas[0]))]


def convert(params: dict, model_path: str | None = None, scale: float = 100.0) -> tuple[Joint, list[dict]]:
    """Вернуть (скелет, кадры) для core.bvh.write. scale=100 → сантиметры."""
    go, bp, tr = params["global_orient"], params["body_pose"], params["transl"]
    if not (len(go) == len(bp) == len(tr)):
        raise ValueError(f"разная длина последовательностей: {len(go)}/{len(bp)}/{len(tr)}")
    betas = mean_betas(params.get("betas"))

    rest = smpl.rest_from_model(model_path, betas) if model_path else list(smpl.DEFAULT_REST)
    root = smpl.build_skeleton(rest, scale)
    pelvis = rest[0]
    flip = ((1.0, 0.0, 0.0), (0.0, -1.0, 0.0), (0.0, 0.0, -1.0)) if params.get("cam_to_yup") else None

    frames: list[dict] = []
    prev: dict[str, tuple[float, float, float]] = {}
    for f in range(len(go)):
        pose = list(bp[f])
        if len(pose) < 63:
            raise ValueError(f"кадр {f}: body_pose длиной {len(pose)}, нужно ≥63")
        rotvecs = [go[f]] + [pose[3 * i: 3 * i + 3] for i in range(smpl.N_JOINTS - 1)]
        frame: dict = {}
        for name, rv in zip(smpl.NAMES, rotvecs):
            m = axis_angle_to_matrix(rv)
            if flip and name == "Hips":
                m = matmul(flip, m)
            e = continuous_euler(matrix_to_euler_zxy(m), prev.get(name))
            prev[name] = e
            frame[name] = {"rot": e}
        pos = tuple(pelvis[k] + float(tr[f][k]) for k in range(3))
        if flip:
            pos = mat_vec(flip, pos)
        frame["Hips"]["pos"] = tuple(c * scale for c in pos)
        frames.append(frame)
    return root, frames

In [ ]:
%%writefile /content/mocapgate_core/gvhmr.py
"""MoCapGate — адаптер GVHMR (https://github.com/zju3dv/GVHMR, некоммерческая лицензия).

GVHMR запускается отдельно (нужен CUDA-GPU: Colab, HF Space, облако):
    python tools/demo/demo.py --video клип.mp4 [-s]
и кладёт результат в outputs/demo/<клип>/hmr4d_results.pt. Там словарь:
    smpl_params_global / smpl_params_incam: body_pose (L,63), betas (L,10),
                                            global_orient (L,3), transl (L,3)
    K_fullimg (L,3,3)
Глобальная система GVHMR — Y вверх (по гравитации), как в BVH.

Читаем три формата:
    .pt   — оригинал, нужен torch (CPU-версии достаточно);
    .npz  — нужен numpy;
    .json — только stdlib: формат «дубль MoCapGate» (colab/MoCapGate_GVHMR.ipynb,
            tools/gvhmr_export.py) — те же ключи плюс fps исходника и сведения о видео.
"""
from __future__ import annotations
import json
from pathlib import Path

KEYS = ("global_orient", "body_pose", "transl", "betas")
SPACES = ("global", "incam")


def _tolist(x):
    return x.tolist() if hasattr(x, "tolist") else x


def _pick(data: dict, space: str) -> dict:
    """Вытащить параметры нужного пространства из любой из раскладок."""
    nested = data.get(f"smpl_params_{space}")
    if isinstance(nested, dict):
        src = nested
    else:  # плоская раскладка: "smpl_params_global.body_pose" или просто "body_pose"
        pref = f"smpl_params_{space}."
        src = {k[len(pref):]: v for k, v in data.items() if k.startswith(pref)} or data
    missing = [k for k in KEYS[:3] if k not in src]
    if missing:
        raise KeyError(f"в результатах GVHMR нет {missing} (пространство '{space}'); ключи: {sorted(data)}")
    out = {k: _tolist(src[k]) for k in KEYS if k in src}
    # torch/numpy могут отдать лишнюю batch-ось (1, L, C)
    for k in KEYS[:3]:
        if out[k] and isinstance(out[k][0], list) and out[k][0] and isinstance(out[k][0][0], list):
            out[k] = out[k][0]
    return out


def _read(path: str) -> dict:
    p = Path(path)
    ext = p.suffix.lower()
    if ext == ".json":
        data = json.loads(p.read_text(encoding="utf-8"))
    elif ext == ".npz":
        import numpy as np  # type: ignore
        z = np.load(p, allow_pickle=True)
        data = {k: (z[k].item() if z[k].dtype == object and z[k].shape == () else z[k]) for k in z.files}
    elif ext == ".pt":
        try:
            import torch  # type: ignore
        except ImportError:
            raise RuntimeError(
                "для .pt нужен torch (pip install torch) — или конвертируй в JSON рядом с GVHMR: "
                "python tools/gvhmr_export.py hmr4d_results.pt"
            ) from None
        data = torch.load(p, map_location="cpu", weights_only=False)
    else:
        raise ValueError(f"неизвестный формат результатов GVHMR: {ext} (ожидаю .pt/.npz/.json)")
    return data


def from_data(data: dict, space: str = "global") -> dict:
    if space not in SPACES:
        raise ValueError(f"space должен быть одним из {SPACES}")
    params = _pick(data, space)
    params["cam_to_yup"] = space == "incam"  # камерная система OpenCV: Y вниз — перевернём в конвертере
    # для оверлея-проверки поверх видео нужны параметры в системе камеры и интринсики
    if space == "global" and isinstance(data.get("smpl_params_incam"), dict):
        params["incam"] = _pick(data, "incam")
    K = data.get("K_fullimg")
    if K is not None:
        K = _tolist(K)
        params["K"] = K[0] if K and isinstance(K[0][0], list) else K  # (L,3,3) → первого кадра
    if data.get("fps"):
        params["fps"] = float(_tolist(data["fps"]))
    if isinstance(data.get("video"), dict):
        params["video"] = data["video"]
    return params


def load_people(path: str, space: str = "global") -> list[dict]:
    data = _read(path)
    if "people" not in data:
        return [{"id": 1, "params": from_data(data, space)}]
    if not isinstance(data["people"], list) or not data["people"]:
        raise ValueError("в результатах нет участников")
    result, seen = [], set()
    for index, person in enumerate(data["people"], 1):
        pid = person.get("id", index)
        if not isinstance(pid, int) or isinstance(pid, bool) or pid < 1 or pid in seen:
            raise ValueError("ID участников должны быть уникальными положительными числами")
        seen.add(pid)
        params = from_data({**{k: data[k] for k in ("fps", "video") if k in data}, **person}, space)
        result.append({"id": pid, "params": params,
                       **{k: person[k] for k in ("start_frame", "end_frame") if k in person}})
    counts = {len(p["params"]["transl"]) for p in result}
    if len(counts) != 1 or not next(iter(counts)):
        raise ValueError("участники должны иметь одинаковую длину общего таймлайна")
    if len({p["params"].get("fps", 30) for p in result}) != 1:
        raise ValueError("участники должны иметь одинаковую частоту кадров")
    return result


def load(path: str, space: str = "global") -> dict:
    """Legacy single-person API; scenes use load_people to preserve every track."""
    return load_people(path, space)[0]["params"]

In [ ]:
#@title Шаг 3. Модель тела SMPL-X (Drive → MoCapGate/ или загрузка)
import shutil, numpy as np
from pathlib import Path
dst = GV / "inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz"
dst.parent.mkdir(parents=True, exist_ok=True)
drive_dir = Path("/content/drive/MyDrive/MoCapGate")
if not dst.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        found = sorted(drive_dir.glob("SMPLX_NEUTRAL*.npz")) if drive_dir.exists() else []
        if found:
            shutil.copy(found[0], dst); print("Взял из Drive:", found[0])
    except Exception as e:
        print("Drive не подключён:", e)
if not dst.exists():
    from google.colab import files
    print("Загрузите SMPLX_NEUTRAL.npz (smpl-x.is.tue.mpg.de → SMPL-X v1.1 NPZ)")
    up = files.upload()
    name = next(iter(up))
    Path(name).replace(dst)
m = np.load(dst, allow_pickle=True)
missing = [k for k in ("v_template", "shapedirs", "J_regressor") if k not in m.files]
if missing:
    dst.unlink(); raise SystemExit(f"Это не модель SMPL-X: нет {missing}. Нужен SMPLX_NEUTRAL.npz")
print("SMPL-X на месте:", dst, m["v_template"].shape)

In [ ]:
#@title Шаг 4. Видео
use_example = False  #@param {type:"boolean"}
#@markdown ↑ взять пример GVHMR (теннис) вместо своего видео
from pathlib import Path
import subprocess, json
IN = Path("/content/in"); IN.mkdir(exist_ok=True)
if use_example:
    src = GV / "docs/example_video/tennis.mp4"
else:
    from google.colab import files
    print("Загрузите видео (mp4 / mov / webm)")
    up = files.upload()
    src = IN / next(iter(up))
    Path(next(iter(up))).replace(src)
name = "".join(c if c.isalnum() or c in "-_" else "_" for c in src.stem)[:60] or "take"
video = IN / f"{name}.mp4"
probe = json.loads(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
    "stream=width,height,avg_frame_rate,nb_frames", "-of", "json", str(src)], capture_output=True, text=True).stdout)["streams"][0]
num, den = (probe.get("avg_frame_rate") or "30/1").split("/")
src_fps = float(num) / float(den or 1)
print(f"{src.name}: {probe['width']}×{probe['height']}, {src_fps:.2f} fps")
# GVHMR ждёт 30 fps: приводим частоту и кодек (webm/HEVC → H.264)
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-vf", f"fps={target_fps}", "-c:v", "libx264",
                "-crf", "18", "-pix_fmt", "yuv420p", "-an", str(video)], check=True)
print("Готово к обработке:", video)

In [ ]:
#@title Шаг 5. GVHMR: поза и траектория
RUNNER_CODE = '"""MoCapGate — запуск GVHMR без рендера (Colab и свой компьютер с NVIDIA GPU).\n\n    <python окружения GVHMR> gvhmr_runner.py <папка GVHMR> <видео 30 fps> <папка вывода> [--static] [--f-mm N]\n\nВыполняется ИНТЕРПРЕТАТОРОМ GVHMR (там torch/hydra/pytorch3d), поэтому без импортов MoCapGate.\nПовторяет предсказание из tools/demo/demo.py, но пропускает рендер: тогда нужна только модель\nSMPL-X (рендер требует ещё SMPL .pkl). Печатает «RESULT <путь к дублю JSON>».\n"""\nimport argparse\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom copy import deepcopy\n\n\ndef track_configs(D, cfg, max_people, torch, fps=30):\n    """Seed GVHMR\'s bbox cache per YOLO ID instead of selecting its top track."""\n    from hmr4d.utils.seq_utils import (frame_id_to_mask, rearrange_by_mask,\n                                      get_frame_id_list_from_mask, linear_interpolate_frame_ids)\n    from hmr4d.utils.net_utils import moving_average_smooth\n    tracker = D.Tracker()\n    history = tracker.track(cfg.video_path)\n    frame_ids, boxes, ordered = tracker.sort_track_length(history, cfg.video_path)\n    del tracker\n    result = []\n    length = len(history)\n    for pid in ordered:\n        # Require a quarter-second of evidence before exporting a new person.\n        # A few tracker observations at the frame edge are often a split ID.\n        if len(frame_ids[pid]) < min(max(3, round(fps*.25)), length):\n            continue\n        person_cfg = deepcopy(cfg)\n        # Resolve interpolation before changing folders; every person shares the same video/timebase.\n        paths = {k: str(v) for k, v in cfg.paths.items()}\n        folder = Path(cfg.output_dir) / f"person_{pid}"\n        folder.mkdir(parents=True, exist_ok=True)\n        person_cfg.output_dir = folder.as_posix()\n        person_cfg.preprocess_dir = folder.as_posix()\n        person_cfg.video_path = cfg.video_path\n        for key, value in paths.items():\n            person_cfg.paths[key] = (folder / Path(value).name).as_posix()\n        ids = torch.tensor(frame_ids[pid])\n        mask = frame_id_to_mask(ids, length)\n        bbx = rearrange_by_mask(torch.tensor(boxes[pid]), mask)\n        bbx = linear_interpolate_frame_ids(bbx, get_frame_id_list_from_mask(~mask))\n        bbx = moving_average_smooth(moving_average_smooth(bbx, window_size=5, dim=0), window_size=5, dim=0)\n        torch.save({"bbx_xyxy": bbx.float(),\n                    "bbx_xys": D.get_bbx_xys_from_xyxy(bbx, base_enlarge=1.2).float()}, person_cfg.paths.bbx)\n        result.append((int(pid), person_cfg, min(frame_ids[pid]), max(frame_ids[pid])))\n        if len(result) >= max_people:\n            break\n    if not result:\n        raise RuntimeError("GVHMR не нашёл участников")\n    return result\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("gvhmr_dir")\n    ap.add_argument("video")\n    ap.add_argument("out_dir")\n    ap.add_argument("--static", action="store_true")\n    ap.add_argument("--f-mm", type=int, default=0)\n    ap.add_argument("--fps", type=float, default=30.0)\n    ap.add_argument("--max-people", type=int, choices=range(1, 9), default=4)\n    a = ap.parse_args()\n\n    gv = os.path.abspath(a.gvhmr_dir)\n    video = Path(a.video).resolve().as_posix()\n    out_dir = Path(a.out_dir).resolve()\n    sys.path[:0] = [gv, os.path.join(gv, "tools", "demo")]\n    os.chdir(gv)\n    sys.argv = ["demo.py", "--video", video, "--output_root", out_dir.as_posix()] + \\\n        (["-s"] if a.static else []) + (["--f_mm", str(a.f_mm)] if a.f_mm else [])\n\n    import hydra  # noqa: E402\n    import torch  # noqa: E402\n    import demo as D  # noqa: E402  tools/demo/demo.py\n\n    cfg = D.parse_args_to_cfg()\n    print("STAGE preprocess", flush=True)\n    tracks = track_configs(D, cfg, a.max_people, torch, a.fps) if a.max_people > 1 else [(1, cfg, 0, None)]\n    for pid, person_cfg, _, _ in tracks:\n        print(f"STAGE preprocess person {pid}", flush=True)\n        D.run_preprocess(person_cfg)\n        torch.cuda.empty_cache()\n    print("STAGE predict", flush=True)\n    model = hydra.utils.instantiate(cfg.model, _recursive_=False)\n    model.load_pretrained_model(cfg.ckpt_path)\n    model = model.eval().cuda()\n    keys = ("global_orient", "body_pose", "transl", "betas")\n    take = {"format": "mocapgate.scene/1", "source": "gvhmr", "fps": a.fps, "people": [],\n            "options": {"static_camera": a.static, "focal_mm": a.f_mm or None}}\n    for pid, person_cfg, start, end in tracks:\n        print(f"STAGE predict person {pid}", flush=True)\n        data = D.load_data_dict(person_cfg)\n        with torch.no_grad():\n            pred = D.detach_to_cpu(model.predict(data, static_cam=cfg.static_cam))\n        torch.save(pred, person_cfg.paths.hmr4d_results)\n        person = {"id": pid, "start_frame": start,\n                  "end_frame": end if end is not None else len(pred["smpl_params_global"]["transl"])-1}\n        for space in ("smpl_params_global", "smpl_params_incam"):\n            person[space] = {k: pred[space][k].float().tolist() for k in keys if k in pred[space]}\n        person["K_fullimg"] = pred["K_fullimg"][0].float().tolist()\n        take["people"].append(person)\n        del data, pred\n        torch.cuda.empty_cache()\n    out = str(out_dir / "result.mocapgate.json")\n    with open(out, "w", encoding="utf-8") as f:\n        json.dump(take, f)\n    print("RESULT", out, flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'

import subprocess, sys
runner = Path("/content/gvhmr_runner.py")
runner.write_text(RUNNER_CODE)
out_dir = Path("/content/out") / name
cmd = [str(PY), str(runner), str(GV), str(video), str(out_dir), "--fps", str(target_fps)]
cmd += ["--max-people", str(max_people)]
cmd += ["--static"] if static_camera else []
cmd += ["--f-mm", str(focal_mm)] if focal_mm else []
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
if p.returncode != 0:
    print(p.stderr[-6000:]); raise SystemExit("GVHMR завершился с ошибкой — текст выше")
result_json = next(l.split(" ", 1)[1] for l in p.stdout.splitlines() if l.startswith("RESULT "))
print("Результат GVHMR:", result_json)

In [ ]:
#@title Шаг 6. BVH + дубль MoCapGate → скачать
import sys, json, shutil, zipfile
sys.path.insert(0, "/content")
from mocapgate_core import gvhmr as mg_gvhmr, smpl_bvh, bvh as mg_bvh
OUT = Path("/content/out") / name; OUT.mkdir(parents=True, exist_ok=True)
take = json.loads(Path(result_json).read_text())
take["video"] = {"name": src.name, "width": int(probe["width"]), "height": int(probe["height"]), "source_fps": src_fps}
take_path = OUT / f"{name}.mocapgate.json"
take_path.write_text(json.dumps(take))
bvh_paths = []
for person in mg_gvhmr.load_people(str(take_path), "global"):
    params = person["params"]
    skel, frames = smpl_bvh.convert(params, str(GV / "inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz"))
    bvh_path = OUT / f"{name}_person_{person['id']}.bvh"
    bvh_path.write_text(mg_bvh.write(skel, frames, 1 / target_fps))
    bvh_paths.append(bvh_path)
shutil.copy(video, OUT / f"{name}.mp4")  # видео 30 fps — для проверки в Studio (оверлей кадр в кадр)
zip_path = Path("/content") / f"{name}_mocapgate.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (take_path, *bvh_paths, OUT / f"{name}.mp4"):
        z.write(f, f.name)
print(f"{len(bvh_paths)} участников, {len(frames)} кадров → BVH + {take_path.name}")
if save_to_drive and Path("/content/drive/MyDrive").exists():
    d = Path("/content/drive/MyDrive/MoCapGate/out"); d.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, d / zip_path.name); print("Сохранено в Drive:", d / zip_path.name)
from google.colab import files
files.download(str(zip_path))

In [ ]:
#@title Шаг 7. Быстрый просмотр скелета
import matplotlib.pyplot as plt
from mocapgate_core import smpl
g = params
idx = [int(i * (len(g["transl"]) - 1) / 4) for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, i in zip(axes, idx):
    P = smpl.fk(g["global_orient"][i], g["body_pose"][i], g["transl"][i])
    for j, par in enumerate(smpl.PARENTS):
        if par >= 0:
            ax.plot([P[par][0], P[j][0]], [P[par][1], P[j][1]], "-", lw=2,
                    color="tab:red" if smpl.NAMES[j].startswith("Left") else "tab:blue" if smpl.NAMES[j].startswith("Right") else "k")
    ax.set_aspect("equal"); ax.set_title(f"кадр {i}"); ax.invert_xaxis()
plt.suptitle("Скелет спереди (красный — левая сторона)"); plt.show()

Дальше: перетащите архив (или `.mocapgate.json` вместе с `.mp4`) в **MoCapGate Studio** — там проверка поверх видео, 3D-просмотр, сглаживание и экспорт в FBX для Maya и движков.